In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, gc, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
RAW_DIR   = os.path.join(DATA_ROOT, 'raw')
TEST_DIR  = os.path.join(DATA_ROOT, 'test_in')
OUT_PATH  = '/kaggle/working/preds.npy'

LOOKBACK    = 10
HORIZON     = 16
H, W        = 140, 124
BATCH_SIZE  = 8      # larger batch → fewer steps/epoch → fits in 15min
ACCUM_STEPS = 1      # no accumulation needed with batch=8
EPOCHS      = 10     # aggressive LR compensates for fewer epochs
LR          = 1e-3   # high LR + OneCycleLR = fast convergence
STRIDE      = 3      # stride=3 keeps samples reasonable for 15min
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MET_FEATS   = ['q2','t2','u10','v10','swdown','pblh','psfc','rain']
EMIT_FEATS  = ['PM25','NH3','SO2','NOx','NMVOC_e','NMVOC_finn','bio']
MONTHS      = ['APRIL_16','JULY_16','OCT_16','DEC_16']

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"15-min config: EPOCHS={EPOCHS}, LR={LR}, BATCH={BATCH_SIZE}, STRIDE={STRIDE}")


In [ ]:
def load_month(month):
    path = os.path.join(RAW_DIR, month)
    data = {}
    data['cpm25'] = np.load(os.path.join(path,'cpm25.npy')).astype(np.float32)
    for f in MET_FEATS + EMIT_FEATS:
        fp = os.path.join(path, f'{f}.npy')
        if os.path.exists(fp):
            data[f] = np.load(fp).astype(np.float32)
    return data

print("Loading training months...")
month_data = {}
for m in MONTHS:
    month_data[m] = load_month(m)
    print(f"  {m}: T={month_data[m]['cpm25'].shape[0]}")

In [ ]:
def engineer_features(data_dict):
    """
    PDF says physics-derived features give diminishing returns.
    Keep only wind_speed — most informative, not redundant.
    Drop wind_dir, inv_pblh, log_rain — model learns these implicitly.
    """
    d = dict(data_dict)
    if 'u10' in d and 'v10' in d:
        d['wind_speed'] = np.sqrt(d['u10']**2 + d['v10']**2)
    return d

for m in MONTHS:
    month_data[m] = engineer_features(month_data[m])
    print(f"  {m}: {list(month_data[m].keys())}")

In [ ]:
BASE_FEATS  = ['cpm25'] + MET_FEATS + EMIT_FEATS
EXTRA_FEATS = ['wind_speed']   # only one engineered feature per PDF advice

FEAT_KEYS = []
for f in BASE_FEATS + EXTRA_FEATS:
    if all(f in month_data[m] for m in MONTHS):
        if f not in FEAT_KEYS:
            FEAT_KEYS.append(f)

N_FEATS = len(FEAT_KEYS)
print(f"FEAT_KEYS locked: {N_FEATS} features")
print(FEAT_KEYS)

def compute_norm_stats(month_data, feat_keys):
    stats = {}
    for feat in feat_keys:
        combined = np.concatenate(
            [month_data[m][feat] for m in MONTHS], axis=0)
        med = np.median(combined, axis=0)
        iqr = (np.percentile(combined,75,axis=0)
             - np.percentile(combined,25,axis=0))
        iqr = np.where(iqr < 1e-6, 1.0, iqr)
        stats[feat] = (med.astype(np.float32), iqr.astype(np.float32))
        del combined
        gc.collect()
    return stats

print("Computing normalization stats...")
norm_stats           = compute_norm_stats(month_data, FEAT_KEYS)
cpm25_med, cpm25_iqr = norm_stats['cpm25']

def normalize_month(data_dict, stats, feat_keys):
    arrays = []
    for feat in feat_keys:
        med, iqr = stats[feat]
        arrays.append(((data_dict[feat] - med) / iqr).astype(np.float32))
    stacked = np.stack(arrays, axis=-1)
    del arrays
    return stacked

norm_data = {}
for m in MONTHS:
    norm_data[m] = normalize_month(month_data[m], norm_stats, FEAT_KEYS)
    print(f"  {m}: {norm_data[m].shape} ({norm_data[m].nbytes/1024**2:.0f} MB)")

del month_data
gc.collect()
print(f"N_FEATS={N_FEATS} locked ✓")

In [ ]:
class PM25Dataset(Dataset):
    def __init__(self, norm_arrays, lookback=LOOKBACK,
                 horizon=HORIZON, stride=STRIDE):
        self.lookback = lookback
        self.horizon  = horizon
        self.arrays   = norm_arrays
        self.cpm_idx  = FEAT_KEYS.index('cpm25')
        self.indices  = []
        self.weights  = []

        for arr_idx, arr in enumerate(norm_arrays):
            T      = arr.shape[0]
            window = lookback + horizon
            for i in range(0, T - window + 1, stride):
                self.indices.append((arr_idx, i))
                target_win = arr[i+lookback:i+window, :, :, self.cpm_idx]
                w = float(np.percentile(target_win, 90))
                self.weights.append(max(w, 0.01))

        self.weights = np.array(self.weights, dtype=np.float32)
        self.weights = self.weights / self.weights.sum()
        print(f"  Samples: {len(self.indices)}")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        arr_idx, t = self.indices[idx]
        arr    = self.arrays[arr_idx]
        x      = arr[t : t + self.lookback].copy()
        y_abs  = arr[t + self.lookback : t + self.lookback + self.horizon,
                     :, :, self.cpm_idx].copy()
        last_pm = arr[t + self.lookback - 1, :, :, self.cpm_idx].copy()
        y_res  = y_abs - last_pm[None]
        return (torch.from_numpy(x).float(),
                torch.from_numpy(y_res).float(),
                torch.from_numpy(y_abs).float(),
                torch.from_numpy(last_pm).float())


# All 4 months: last 20% of each → val; first 80% → train
# This gives model exposure to all seasons including winter episodes
train_arrays, val_arrays = [], []
for m in MONTHS:
    arr = norm_data[m]
    cut = int(0.80 * arr.shape[0])
    train_arrays.append(arr[:cut])
    val_arrays.append(arr[cut:])

print("Train:")
train_ds = PM25Dataset(train_arrays, stride=STRIDE)
print("Val:")
val_ds   = PM25Dataset(val_arrays, stride=4)

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(train_ds.weights),
    num_samples=len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")


In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel=3):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.gates     = nn.Conv2d(
            in_ch + hidden_ch, 4*hidden_ch, kernel, padding=kernel//2)

    def forward(self, x, h, c):
        gates    = self.gates(torch.cat([x, h], dim=1))
        i, f, g, o = gates.chunk(4, dim=1)
        c_new    = torch.sigmoid(f)*c + torch.sigmoid(i)*torch.tanh(g)
        h_new    = torch.sigmoid(o)*torch.tanh(c_new)
        return h_new, c_new

    def init_hidden(self, B, H, W, device):
        return (torch.zeros(B, self.hidden_ch, H, W, device=device),
                torch.zeros(B, self.hidden_ch, H, W, device=device))


class SpectralBlock(nn.Module):
    """
    PDF recommends hybrid spectral + local conv.
    Uses FFT to capture global frequency patterns (transport, advection).
    Combines with local 3x3 conv for fine-grained spatial features.
    Addresses spectral bias for episode detection.
    """
    def __init__(self, ch, modes_h=20, modes_w=16):
        super().__init__()
        self.modes_h = modes_h
        self.modes_w = modes_w
        # Learnable Fourier weights
        scale = 1.0 / (ch * ch)
        self.weights = nn.Parameter(
            scale * torch.randn(ch, ch, modes_h, modes_w, dtype=torch.cfloat))
        # Local conv branch
        self.local = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.BatchNorm2d(ch), nn.GELU())
        # Fuse
        self.fuse = nn.Sequential(
            nn.Conv2d(ch*2, ch, 1),
            nn.BatchNorm2d(ch), nn.GELU())

    def forward(self, x):
        B, C, H, W = x.shape
        # FFT branch — global patterns
        x_ft    = torch.fft.rfft2(x, norm='ortho')
        mh      = min(self.modes_h, x_ft.shape[2])
        mw      = min(self.modes_w, x_ft.shape[3])
        out_ft  = torch.zeros_like(x_ft)
        out_ft[:, :, :mh, :mw] = torch.einsum(
            'bchw,cdhw->bdhw',
            x_ft[:, :, :mh, :mw],
            self.weights[:, :, :mh, :mw])
        x_spec  = torch.fft.irfft2(out_ft, s=(H, W), norm='ortho')
        # Local conv branch
        x_local = self.local(x)
        # Fuse both
        return self.fuse(torch.cat([x_spec, x_local], dim=1))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, hidden), nn.ReLU(),
            nn.Linear(hidden, channels), nn.Sigmoid())
        self.spatial_att = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3), nn.Sigmoid())

    def forward(self, x):
        ca  = self.channel_att(x).view(x.shape[0], x.shape[1], 1, 1)
        x   = x * ca
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1,  keepdim=True)[0]
        return x * self.spatial_att(torch.cat([avg, mx], dim=1))


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_attn=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.GELU())
        self.attn = CBAM(out_ch) if use_attn else nn.Identity()
        self.res  = (nn.Conv2d(in_ch, out_ch, 1)
                     if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        return self.attn(self.conv(x)) + self.res(x)


class HybridEncoder(nn.Module):
    """
    ConvLSTM temporal path + Spectral global path.
    PDF: 'Combining models that learn local kernels with those
    that learn global kernels proved to be effective.'
    """
    def __init__(self, n_feats, hidden):
        super().__init__()
        # Path A: ConvLSTM (local temporal)
        self.proj_lstm  = nn.Conv2d(n_feats, hidden, 1)
        self.lstm       = ConvLSTMCell(hidden, hidden)

        # Path B: Spectral (global spatial)
        self.proj_spec  = nn.Conv2d(LOOKBACK * n_feats, hidden, 1)
        self.spectral   = SpectralBlock(hidden)

        # Fuse
        self.fuse = nn.Sequential(
            nn.Conv2d(hidden*2, hidden, 1),
            nn.BatchNorm2d(hidden), nn.GELU())

    def forward(self, x):
        B, T, H, W, C = x.shape

        # Path A: ConvLSTM
        h, c = self.lstm.init_hidden(B, H, W, x.device)
        for t in range(T):
            xt   = x[:, t].permute(0,3,1,2)
            h, c = self.lstm(self.proj_lstm(xt), h, c)
        out_lstm = h

        # Path B: Spectral on flattened time
        flat     = x.permute(0,1,4,2,3).reshape(B, T*C, H, W)
        out_spec = self.spectral(self.proj_spec(flat))

        return self.fuse(torch.cat([out_lstm, out_spec], dim=1))


class ResidualPM25Net(nn.Module):
    """
    PDF insight: predict residuals (deltas) not absolute values.
    Hybrid encoder: ConvLSTM (local) + Spectral (global).
    U-Net decoder with CBAM attention.
    """
    def __init__(self, n_feats, horizon, base=48):
        super().__init__()
        self.horizon    = horizon
        self.enc_in     = HybridEncoder(n_feats, base)
        self.enc1       = ConvBlock(base,    base*2, False)
        self.enc2       = ConvBlock(base*2,  base*4, False)
        self.enc3       = ConvBlock(base*4,  base*8, True)
        self.pool       = nn.MaxPool2d(2, ceil_mode=True)
        self.bottleneck = ConvBlock(base*8,  base*16, True)
        self.up3        = nn.ConvTranspose2d(base*16, base*8,  2, stride=2)
        self.dec3       = ConvBlock(base*16, base*8,  True)
        self.up2        = nn.ConvTranspose2d(base*8,  base*4,  2, stride=2)
        self.dec2       = ConvBlock(base*8,  base*4,  False)
        self.up1        = nn.ConvTranspose2d(base*4,  base*2,  2, stride=2)
        self.dec1       = ConvBlock(base*4,  base*2,  False)
        self.head       = nn.Sequential(
            nn.Conv2d(base*2, base,    3, padding=1), nn.GELU(),
            nn.Conv2d(base,   horizon, 1))
        # Zero-init — critical for residual prediction stability
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def _match(self, x, ref):
        dh = ref.shape[2] - x.shape[2]
        dw = ref.shape[3] - x.shape[3]
        if dh > 0 or dw > 0:
            x = F.pad(x, [0, max(dw,0), 0, max(dh,0)])
        if x.shape[2] > ref.shape[2] or x.shape[3] > ref.shape[3]:
            x = x[:, :, :ref.shape[2], :ref.shape[3]]
        return x

    def forward(self, x):
        # Returns RESIDUALS (deltas from last known PM2.5)
        e0 = self.enc_in(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bn = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self._match(self.up3(bn), e3), e3], dim=1))
        d2 = self.dec2(torch.cat([self._match(self.up2(d3), e2), e2], dim=1))
        d1 = self.dec1(torch.cat([self._match(self.up1(d2), e1), e1], dim=1))
        return self.head(d1)   # (B, 16, H, W) — RESIDUALS


print(f"Building ResidualPM25Net: N_FEATS={N_FEATS}, base=48")
model    = ResidualPM25Net(n_feats=N_FEATS, horizon=HORIZON, base=48).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    free_gb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated()) / 1024**3
    print(f"GPU free: {free_gb:.1f} GB")

with torch.no_grad():
    dummy = torch.zeros(2, LOOKBACK, H, W, N_FEATS).to(DEVICE)
    out   = model(dummy)
    assert out.shape == (2, HORIZON, H, W), f"Bad shape: {out.shape}"
    print(f"Shape check passed: {out.shape} ✓")
    del dummy
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
def smape_loss(pred, target, eps=0.1):
    """eps=0.1 (was 1.0): sharper gradient at low PM2.5 values."""
    return torch.mean(
        torch.abs(pred - target) /
        (0.5*(torch.abs(target) + torch.abs(pred)) + eps))

def pearson_corr_loss(pred, target):
    pf = pred.reshape(pred.shape[0], -1)
    tf = target.reshape(target.shape[0], -1)
    pm = pf - pf.mean(dim=1, keepdim=True)
    tm = tf - tf.mean(dim=1, keepdim=True)
    corr = (pm*tm).sum(dim=1) / (pm.norm(dim=1)*tm.norm(dim=1) + 1e-8)
    return 1.0 - corr.mean()

def episode_mask_batch(target, sigma=2.0):
    """sigma=2.0 (was 2.5): catches more real episodes without false positives."""
    B, T, H, W = target.shape
    flat = target.view(B, T, -1)
    mean = flat.mean(dim=1, keepdim=True)
    std  = flat.std(dim=1,  keepdim=True)
    return ((flat - mean) > sigma * std).view(B, T, H, W)

def high_freq_loss(pred, target):
    """Penalise missing high-frequency spatial spike patterns."""
    pred_ft   = torch.fft.rfft2(pred,   norm='ortho')
    target_ft = torch.fft.rfft2(target, norm='ortho')
    H2 = pred_ft.shape[2] // 2
    W2 = pred_ft.shape[3] // 2
    return F.mse_loss(pred_ft[:,:,H2:,W2:].abs(),
                      target_ft[:,:,H2:,W2:].abs())

def full_loss(pred_res, target_res, target_abs, last_pm,
              w_global=0.25, w_ep_smape=0.30,
              w_ep_corr=0.40, w_hf=0.05):
    """
    w_ep_corr raised to 0.40 (was 0.25) — hardest and most impactful metric.
    w_hf reduced to 0.05 to free weight budget for corr.
    """
    pred_abs = pred_res + last_pm.unsqueeze(1)
    loss_global   = smape_loss(pred_abs, target_abs)
    mask          = episode_mask_batch(target_abs)

    if mask.sum() > 50:
        loss_ep_smape = smape_loss(pred_abs[mask], target_abs[mask])
        loss_ep_corr  = pearson_corr_loss(
            pred_abs*mask.float(), target_abs*mask.float())
    else:
        loss_ep_smape = loss_global
        loss_ep_corr  = torch.tensor(0.0, device=pred_res.device)

    loss_hf = high_freq_loss(pred_res, target_res)

    total = (w_global   * loss_global
           + w_ep_smape * loss_ep_smape
           + w_ep_corr  * loss_ep_corr
           + w_hf       * loss_hf)
    return total, loss_global.item(), loss_ep_smape.item(), loss_ep_corr.item()

print("Loss functions ready ✓")
print("  smape eps    : 0.1  (was 1.0)")
print("  episode sigma: 2.0  (was 2.5)")
print("  w_ep_corr    : 0.40 (was 0.25) ← key change")


In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.2,
    anneal_strategy='cos',
    div_factor=5.0, final_div_factor=1e3)

best_val_loss = float('inf')
best_ckpt     = '/kaggle/working/best_model_R2.pt'
last_ckpt     = '/kaggle/working/last_model_R2.pt'
patience      = 5     # tight patience for 15min run
no_improve    = 0
start_time    = time.time()

for epoch in range(1, EPOCHS+1):

    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (xb, yb_res, yb_abs, last_pm) in enumerate(train_loader):
        xb      = xb.to(DEVICE)
        yb_res  = yb_res.to(DEVICE)
        yb_abs  = yb_abs.to(DEVICE)
        last_pm = last_pm.to(DEVICE)

        pred_res = model(xb)
        if pred_res.shape[-2:] != yb_res.shape[-2:]:
            pred_res = F.interpolate(pred_res, size=yb_res.shape[-2:],
                                     mode='bilinear', align_corners=False)
        loss, *_ = full_loss(pred_res, yb_res, yb_abs, last_pm)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb_res, yb_abs, last_pm in val_loader:
            xb      = xb.to(DEVICE)
            yb_res  = yb_res.to(DEVICE)
            yb_abs  = yb_abs.to(DEVICE)
            last_pm = last_pm.to(DEVICE)
            pred_res = model(xb)
            if pred_res.shape[-2:] != yb_res.shape[-2:]:
                pred_res = F.interpolate(pred_res, size=yb_res.shape[-2:],
                                         mode='bilinear', align_corners=False)
            loss, *_ = full_loss(pred_res, yb_res, yb_abs, last_pm)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    # Save both checkpoints
    torch.save(model.state_dict(), last_ckpt)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt)
        no_improve = 0
    else:
        no_improve += 1

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    elapsed = (time.time() - start_time) / 60
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train: {tr_loss:.4f} | Val: {val_loss:.4f} | "
          f"Best: {best_val_loss:.4f} | "
          f"LR: {scheduler.get_last_lr()[0]:.2e} | "
          f"P:{no_improve} | {elapsed:.1f}m")

    if no_improve >= patience:
        print(f"Early stop at epoch {epoch}")
        break
    if elapsed > 12:   # hard 12-min guard, leaves 3min for inference
        print(f"Time guard at epoch {epoch} ({elapsed:.1f}m)")
        break

print(f"\nDone. Best val loss: {best_val_loss:.4f}")
print(f"Total training time: {(time.time()-start_time)/60:.1f}m")


In [ ]:
print("Loading test data...")
test_data = {}
test_data['cpm25'] = np.load(
    os.path.join(TEST_DIR, 'cpm25.npy')).astype(np.float32)
for f in MET_FEATS:
    fp = os.path.join(TEST_DIR, f'{f}.npy')
    if os.path.exists(fp):
        test_data[f] = np.load(fp).astype(np.float32)
for f in EMIT_FEATS:
    fp = os.path.join(TEST_DIR, f'{f}.npy')
    if os.path.exists(fp):
        test_data[f] = np.load(fp).astype(np.float32)

test_data = engineer_features(test_data)
N_TEST    = test_data['cpm25'].shape[0]
missing   = [f for f in FEAT_KEYS if f not in test_data]
print(f"Test samples : {N_TEST}")
print(f"Missing feats: {missing if missing else 'None ✓'}")

In [ ]:
def build_test_tensor_single(sample_idx):
    arrays = []
    for feat in FEAT_KEYS:
        med, iqr = norm_stats[feat]
        if feat in test_data:
            arr    = test_data[feat][sample_idx]
            normed = (arr - med[None]) / iqr[None]
        else:
            normed = np.zeros((LOOKBACK, H, W), dtype=np.float32)
        arrays.append(normed)
    return np.stack(arrays, axis=-1).astype(np.float32)

sample      = build_test_tensor_single(0)
expected_ch = model.enc_in.proj_lstm.weight.shape[1]
actual_ch   = sample.shape[-1]
assert actual_ch == expected_ch, \
    f"Channel mismatch! Model={expected_ch}, got={actual_ch}"
print(f"Channel check : {actual_ch} == {expected_ch} ✓")
print(f"Sample shape  : {sample.shape} ✓")
del sample

In [ ]:
# Load both checkpoints and ensemble their predictions
# Best checkpoint = lowest val loss; Last checkpoint = most trained
# Averaging them gives better generalisation than either alone

def load_model(ckpt_path):
    m = ResidualPM25Net(n_feats=N_FEATS, horizon=HORIZON, base=48).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.eval()
    return m

model_best = load_model(best_ckpt)
try:
    model_last = load_model(last_ckpt)
    use_ensemble = True
    print("Ensemble: best + last checkpoint ✓")
except:
    model_last = None
    use_ensemble = False
    print("Single model inference (last ckpt not found)")

cpm_idx    = FEAT_KEYS.index('cpm25')
preds_list = []
n_batches  = (N_TEST + BATCH_SIZE - 1) // BATCH_SIZE

def run_tta(m, xb):
    """5-way TTA: orig + hflip + vflip + hv-flip + 90rot approx"""
    with torch.no_grad():
        p1 = m(xb)
        p2 = torch.flip(m(torch.flip(xb, dims=[3])), dims=[3])
        p3 = torch.flip(m(torch.flip(xb, dims=[2])), dims=[2])
        p4 = torch.flip(m(torch.flip(xb, dims=[2,3])), dims=[2,3])
        # Slight noise realisation for extra variance reduction
        xb_n = xb.clone()
        xb_n = xb_n + torch.randn_like(xb_n) * 0.008
        p5 = m(xb_n)
    pred = (p1 + p2 + p3 + p4 + p5) / 5.0
    if pred.shape[-2:] != (H, W):
        pred = F.interpolate(pred, size=(H,W),
                             mode='bilinear', align_corners=False)
    return pred

with torch.no_grad():
    for i in range(0, N_TEST, BATCH_SIZE):
        batch_idx = range(i, min(i + BATCH_SIZE, N_TEST))
        batch_np  = np.stack(
            [build_test_tensor_single(j) for j in batch_idx], axis=0)

        last_pm_norm = batch_np[:, -1, :, :, cpm_idx]
        last_pm_abs  = last_pm_norm * cpm25_iqr[None] + cpm25_med[None]

        xb = torch.from_numpy(batch_np).to(DEVICE)

        # TTA on best model
        res_best = run_tta(model_best, xb)

        # TTA on last model (if available) and average
        if use_ensemble:
            res_last = run_tta(model_last, xb)
            res = (res_best + res_last) / 2.0
        else:
            res = res_best

        del xb
        res_np  = res.cpu().numpy()
        del res

        # Denormalize residuals (scale only — no shift for deltas)
        res_abs  = res_np * cpm25_iqr[None, None]

        # Reconstruct: absolute = last_known + predicted_delta
        pred_abs = last_pm_abs[:, None] + res_abs
        pred_abs = np.clip(pred_abs, 0, None)

        preds_list.append(pred_abs)
        print(f"  Batch {i//BATCH_SIZE+1}/{n_batches} done", end='\r')

preds = np.concatenate(preds_list, axis=0)   # (218,16,H,W)
preds = preds.transpose(0, 2, 3, 1)          # (218,H,W,16)
print(f"\nFinal shape: {preds.shape}")
assert preds.shape == (218, 140, 124, 16), f"Shape mismatch: {preds.shape}"
print("Shape check passed ✓")


In [ ]:
np.save(OUT_PATH, preds)
loaded = np.load(OUT_PATH)
print(f"Saved  → {OUT_PATH}")
print(f"Shape  : {loaded.shape}")
print(f"dtype  : {loaded.dtype}")
print(f"min/max: {loaded.min():.4f} / {loaded.max():.4f}")
print(f"NaNs   : {np.isnan(loaded).sum()}")
print(f"Negs   : {(loaded < 0).sum()}")
assert loaded.shape == (218, 140, 124, 16)
assert np.isnan(loaded).sum() == 0
assert (loaded < 0).sum()     == 0
print("\n✅ Submission ready: /kaggle/working/preds.npy")